# TSFM Comparison Research — Overall Research Plan

**Project:** Comparing time-series foundation models (TSFMs) for financial forecasting, and using what differentiates them to build a regime-aware adaptive forecaster.

**Models under study:** Kronos, Lag-Llama, TimesFM (foundation models) vs. iTransformer (trained-from-scratch conventional baseline).

**Markets:** S&P 500, Shanghai Composite (SSE), Shenzhen Component (SZSE), Nikkei 225 — chosen to span US, Chinese A-share, and Japanese equity markets.

This notebook lays out the plan as designed, before execution. See `02_progress_and_next_steps.ipynb` for what has actually been done against this plan.


## 1. Motivation

Time-series foundation models (TSFMs) — large models pretrained on broad, cross-domain time-series corpora and applied zero-shot or fine-tuned to a new series — are a recent development, originally proven mostly on general forecasting benchmarks (energy, traffic, weather). Their behavior on **financial** time series specifically is less well understood, and financial series have properties (heavy tails, regime shifts, near-random-walk efficiency) that general-purpose benchmarks don't stress in the same way.

At the same time, no single TSFM is likely to dominate under all conditions. A model that is well-calibrated and accurate during calm, trending markets may degrade sharply during high-volatility or structural-break periods, and vice versa. If that's true, the interesting question isn't just "which model is best on average" but **"which model is best under which conditions, and can that be exploited"**.


## 2. Research Questions

1. **How do Kronos, Lag-Llama, and TimesFM differ in their forecasting performance on financial time series** (accuracy: MAE, RMSE, directional accuracy), compared to each other and to a conventional trained-from-scratch baseline (iTransformer)?
2. **How does the stability or regime of the input time series affect the accuracy, robustness, and uncertainty of each model?** Specifically: do models have different strengths depending on stationarity, volatility, and trend strength of the recent history, and does forecast uncertainty degrade predictably as inputs become less stable?
3. **Can time-series stability information be used to dynamically combine multiple models** — weighting Kronos/Lag-Llama/TimesFM according to current regime — to outperform any single model or a fixed-weight ensemble?


## 3. Methodology — Four Phases

### Phase 1: Unified benchmark
Run all three foundation models and the iTransformer baseline on the same financial series under an identical evaluation protocol (same context length, same forecast horizon, same test windows, same metrics: MAE, RMSE, directional accuracy). This establishes a controlled baseline before asking *why* models differ.

### Phase 2: Regime characterization
For each forecasting window, compute characteristics of the *historical* (input) series:
- **Stationarity** — ADF / KPSS tests (complementary null hypotheses), or a Hurst exponent as a continuous alternative.
- **Volatility** — realized volatility over the window, optionally GARCH-implied conditional volatility.
- **Trend strength** — STL decomposition trend-to-remainder ratio, or ADX.

Bucket windows into regimes (e.g. stable / medium / unstable) and break out Phase 1's performance metrics by regime, to see whether models have different strengths depending on input characteristics.

### Phase 3: Uncertainty and robustness analysis
For probabilistic models (Lag-Llama, TimesFM), use their native quantile/distributional outputs to measure calibration (interval coverage, CRPS, pinball loss). For models without a native distribution (Kronos), estimate uncertainty via repeated stochastic sampling. Test whether input instability (from Phase 2) predicts *forecast* unreliability — i.e. does miscalibration correlate with regime?

### Phase 4: Stability-aware dynamic fusion
Design a fusion method that dynamically reweights Kronos / Lag-Llama / TimesFM according to the *current* regime characteristics (from Phase 2), rather than fixed ensemble weights. Compare against: equal-weight static ensemble, inverse-error static ensemble, best-single-model-in-hindsight (oracle upper bound), and ideally a reimplemented existing regime-aware ensemble method from the literature.


## 4. Methodological safeguards

These were identified as necessary before running any experiments, to avoid results that look meaningful but aren't:

- **Leakage-safe evaluation window.** Kronos, Lag-Llama, and TimesFM were all pretrained on broad corpora with unclear or partial overlap with the specific series under test. A test window chosen *after* all three models' known pretraining cutoffs is required, or any "TSFM beats baseline" result is unfalsifiable (could just be memorization). Documented in the papers/reported cutoffs, Kronos's own held-out window ends ~2025-06-05 — the tightest constraint of the three.
- **Zero-shot vs. fine-tuned, decided explicitly.** iTransformer is trained from scratch on the target series; the three TSFMs are run zero-shot by default. This is an inherent asymmetry (transfer learning vs. supervised learning) that has to be acknowledged in any comparison, and ideally resolved later with a fine-tuned-TSFM condition.
- **Statistical significance, not point-estimate comparison.** Financial-series MAE/RMSE differences between models are frequently not statistically significant given typical sample sizes. Pairwise comparisons should use a **Diebold-Mariano test** (or Model Confidence Set), with a variance correction for the serial correlation introduced by overlapping rolling windows.
- **A naive baseline** (persistence / random walk) should accompany every comparison — for financial series this is often surprisingly hard to beat, especially on directional accuracy where 50% is chance.
- **Multi-market generalization.** A single asset/series makes every finding one draw from a noisy process; testing across multiple markets (different volatility regimes, different pretraining-corpus representation) strengthens any generalizable claim and can itself surface findings (e.g. does a model's edge depend on how well-represented that market was in its pretraining data?).
- **Compute cost as a secondary axis.** Model comparisons that only report accuracy miss that these models differ substantially in parameter count and inference cost; reporting cost alongside accuracy is what makes a comparison practically useful, not just academic.


## 5. Evaluation design (as planned)

- **Context / horizon:** fixed context length and forecast horizon shared identically across all models, so "same input" means the same information content, not just the same array shape.
- **Rolling windows:** step forward through the leakage-safe test period at a fixed stride, producing multiple overlapping forecast windows per market rather than one single evaluation — necessary for any significance testing to be meaningful.
- **Metrics:** MAE, RMSE, directional accuracy (paired with a persistence baseline so directional accuracy can't be misread as skill), and — where available — 80% predictive-interval coverage as a calibration check.
- **No-lookahead discipline for Phase 4:** regime features and fusion weights at time *t* must be computable from information available at *t* only (trailing windows, not centered ones), to avoid silently leaking future information into the fusion weights.


## 6. Deliverables

1. A reusable, dataset-agnostic evaluation harness (not one-off scripts) that can be pointed at any new market with minimal additional work.
2. A statistically-grounded Phase 1 comparison across multiple financial markets.
3. A regime-conditioned performance and calibration breakdown (Phase 2/3).
4. A working stability-aware fusion prototype, benchmarked against static-ensemble and oracle baselines (Phase 4).
5. A written account of *when and why* each model performs well — the interpretive payoff, not just the numbers.
